In [6]:
import pyarrow.parquet as pq
import numpy as np, pandas as pd

In [7]:
def r_precision (recommended, hidden):
    """recommended: array of track indices, ranked hidden: set of track indices"""
    R = len(hidden)
    return len(set(recommended[:R]) & hidden)/ R

In [8]:
# load in validation playlists 
split = pd.read_parquet("../gold/split")
val_pids = split.loc[split["split"]=="validation", "pid"].to_numpy()


In [9]:
val_set = set(val_pids.tolist()) #list of validation pids
pt = pq.read_table(
    "../silver/pid_pos",
    columns = ["pid", "pos", "track_uri"],
    filters=[("pid", "in", val_set)],
).to_pandas()

In [10]:
#mapping track_uris to track index
tidx=pd.read_parquet("../gold/track_index")
pt = pt.merge(tidx, on="track_uri", how="left")
assert pt["track_idx"].notna().all(), "unmapped track_uri"

In [11]:
#we will now deduplicate, keeping the first occurrence positition of every track 
#since some tracks repeat multiple times in a playlist and matrix is binary
pt = pt.sort_values(["pid", "pos"])
pt = pt.drop_duplicates(["pid", "track_idx"], keep="first")

In [ ]:
# dictionary that has playlist ids as keys and track_idx list for that playlist as values
truth = (pt.groupby("pid")["track_idx"]
         .apply(lambda s: s.to_numpy(dtype=np.int64))
         .to_dict())

In [13]:
p = val_pids[0]
print(truth[p][:10])
print(pt[pt["pid"]== p].sort_values("pos").head(10)[["pos","track_idx"]])
#should match after sorting

[2057555 1398431 1378687   15567  788293  468175  782622  884769  532271
 1384313]
        pos  track_idx
128807    0    2057555
542514    1    1398431
658272    2    1378687
625074    3      15567
277322    4     788293
509553    5     468175
360289    6     782622
476603    7     884769
377047    8     532271
95521     9    1384313


In [14]:
print(len(truth))                                        # 10000
lengths = np.array([len(v) for v in truth.values()])
print(lengths.min(), np.median(lengths), lengths.max())
#min tells us the smalled playlist size within our dataset
#max tell us the max tracks within a single playlist

10000
5 48.0 250


In [15]:
meta = pd.read_parquet("../artifacts/tracks.parquet") #meta data for only training data
meta.head()

,track_idx,track_uri,track_name,artist_name,n_playlists,search_key
0,0,spotify:track:0000uJA4xCdxThagdLkkLR,Heart As Cold As Stone,Cherryholmes,1,heart as cold as stone-cherryholmes
1,1,spotify:track:0002yNGLtYSYtc0X6ZnFvp,Muskrat Ramble,Sidney Bechet's Blue Note Jazzmen,6,muskrat ramble-sidney bechet's blue note jazzmen
2,2,spotify:track:00039MgrmLoIzSpuYKurn9,Thas What I Do,Zach Farlow,2,thas what i do-zach farlow
3,3,spotify:track:0003Z98F6hUq7XxqSRM87H,Меня больше тут нет,Жак-Энтони,1,меня больше тут нет-жак-энтони
4,4,spotify:track:0004ExljAge0P5XWn1LXmW,Gita,RPM,0,gita-rpm


In [16]:
in_train = meta["n_playlists"].to_numpy() > 0 #checking

In [ ]:
#given the tracks that are seeded, we want to make sure we are not looking for tracks that were not a part of training
def make_case (tracks, n_seeds):
    """tracks: playlist index array in order. Returns (seeds, hidden) or None is unusable"""
    if len(tracks) <= n_seeds: #no hidden tracks to predict
        return None   
    seeds = tracks [:n_seeds] #give the model the playlist's first n seeds
    hidden = tracks [n_seeds:] #hide everything else
    hidden = hidden[in_train[hidden]] #drops tracks the model cannont predict (not in training)
    if hidden.size == 0:
        return None
    return seeds, set(hidden.tolist())

In [ ]:
#what is the print of this saying?
N_SEEDS = 5
cases = {}
for pid, tracks in truth.items():
    c = make_case(tracks, N_SEEDS)
    if c is not None:
        cases[pid] = c

print(f"{len(cases)} usable of {len(truth)}")
# we get 9952 playlists are 'usable' out of '10000'

9952 usable of 10000
